# MNIST Baseline Walkthrough

This notebook is a step-by-step learning walkthrough. The repository's `.py`
scripts, experiment log, result tables, and summaries are the authoritative
sources for the controlled experiments.

Learning goals:

- load one MNIST batch with the project DataLoader
- inspect image, label, logits, and prediction shapes
- calculate hard-label cross-entropy loss
- perform one correct optimizer update
- understand `zero_grad()`, `backward()`, and `step()`


## Outline

1. Import project code with a portable project-root setup.
2. Set a small walkthrough seed.
3. Load and inspect one MNIST batch.
4. Create the hard-label student model.
5. Run a forward pass and inspect logits.
6. Calculate cross-entropy loss.
7. Perform one optimizer update.
8. Connect the notebook to the formal training script.


## 1. Imports and portable project-root setup

The notebook should work whether it is opened from the repository root or from
the `notebooks/` folder. It should not need a hardcoded local path.


In [1]:
from pathlib import Path
import sys

import torch
from torch import nn
from torch.optim import Adam

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from kd_research.data import create_mnist_loaders
from kd_research.models import SimpleCNNStudentModel

print("project root detected:", PROJECT_ROOT.name)


project root detected: knowledge-distillation-research


## 2. Reproducibility seed for the walkthrough

This seed makes the walkthrough output more consistent when re-run. It is not
the full formal reproducibility protocol used for controlled experiments.


In [2]:
torch.manual_seed(0)
print("walkthrough seed set to 0")


walkthrough seed set to 0


## 3. Load one MNIST batch

A batch is a small group of images and labels. Here we use only one batch so we
can focus on the mechanics rather than full training.


In [3]:
train_loader, test_loader = create_mnist_loaders(
    data_dir=PROJECT_ROOT / "data",
    batch_size=8,
    download=False,
    num_workers=0,
)

images, labels = next(iter(train_loader))

print("image batch shape:", tuple(images.shape))
print("label batch shape:", tuple(labels.shape))
print("sample labels:", labels.tolist())
print("image value range:", round(float(images.min()), 3), "to", round(float(images.max()), 3))


image batch shape: (8, 1, 28, 28)
label batch shape: (8,)
sample labels: [6, 8, 8, 7, 8, 0, 0, 5]
image value range: 0.0 to 1.0


## 4. Inspect image and label shapes

For MNIST, each image is grayscale, so the channel dimension is `1`. The shape
`(8, 1, 28, 28)` means:

- 8 images in the batch
- 1 grayscale channel
- 28 pixel height
- 28 pixel width

The labels are digit classes from 0 to 9.


In [4]:
print("first image shape:", tuple(images[0].shape))
print("first label:", int(labels[0]))


first image shape: (1, 28, 28)
first label: 6


## 5. Create the student model

The hard-label baseline uses the student CNN only. No teacher model and no soft
targets are involved.


In [5]:
student = SimpleCNNStudentModel()
print(student)


SimpleCNNStudentModel(
  (network): Sequential(
    (0): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Flatten(start_dim=1, end_dim=-1)
    (3): Linear(in_features=6272, out_features=10, bias=True)
  )
)


## 6. Run a forward pass

A forward pass sends the images through the model. The output is called
`logits`: raw class scores before softmax.


In [6]:
student.eval()
with torch.inference_mode():
    logits = student(images)

predictions = logits.argmax(dim=1)

print("logits shape:", tuple(logits.shape))
print("predictions shape:", tuple(predictions.shape))
print("first image logits:", [round(float(x), 4) for x in logits[0]])
print("predicted labels:", predictions.tolist())


logits shape: (8, 10)
predictions shape: (8,)
first image logits: [0.0003, 0.1453, 0.0932, 0.0269, 0.1097, 0.0103, -0.0229, -0.2081, 0.0127, -0.0082]
predicted labels: [1, 1, 1, 8, 1, 1, 1, 1]


## 7. Calculate cross-entropy loss

Hard-label training compares the student's logits with the true MNIST labels.
`CrossEntropyLoss` is the normal loss for single-label classification.


In [7]:
student.train()
loss_fn = nn.CrossEntropyLoss()
logits = student(images)
loss = loss_fn(logits, labels)

print("cross-entropy loss:", round(float(loss.item()), 4))


cross-entropy loss: 2.3456


## 8. Create the optimizer

The optimizer controls how model parameters change after gradients are
computed. This project used Adam for the MNIST baseline script.


In [8]:
optimizer = Adam(student.parameters(), lr=0.001)
print("optimizer:", optimizer.__class__.__name__)


optimizer: Adam


## 9. Perform one correct optimizer update

The correct order is:

```python
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

- `optimizer.zero_grad()` clears old gradients.
- `loss.backward()` computes new gradients from the current loss.
- `optimizer.step()` updates the model parameters.

This is only one tiny update, not an official experiment result.


In [9]:
first_parameter = next(student.parameters())
before_update = first_parameter.detach().clone()

optimizer.zero_grad()
logits = student(images)
loss = loss_fn(logits, labels)
loss.backward()
optimizer.step()

after_update = first_parameter.detach().clone()
student_updated = not torch.equal(before_update, after_update)

print("loss used for update:", round(float(loss.item()), 4))
print("student updated:", student_updated)


loss used for update: 2.3456
student updated: True


## 10. Relation to the full baseline script

After the mechanics make sense, the reusable script is the source for full
baseline training:

```powershell
.\.venv\Scripts\python.exe scripts/train_mnist_baseline.py --epochs 1 --batch-size 64 --learning-rate 0.001 --seed 0
```

This notebook demonstrates the training workflow. Official controlled
experiment results are reported in the repository result summaries and
experiment log.


## Mini exercise

Answer in your own words:

1. Why is the image batch shape `(8, 1, 28, 28)`?
2. Why is the logits shape `(8, 10)`?
3. Why is this notebook not Knowledge Distillation?


## What I learned

- A DataLoader gives a batch of images and labels.
- The student model converts images into logits.
- Cross-entropy compares logits with true hard labels.
- Gradients are created by `loss.backward()`.
- Parameters change only after `optimizer.step()`.
- This walkthrough is educational; the formal results live in the scripts,
  logs, tables, and summaries.
